# Lesson 24 Lab — Backend Portability and the ROCm Boundary

**Puzzle:** When shared source, backend targets, architecture tuning, and unmeasured platforms change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates shared source, backend targets, architecture tuning, and unmeasured platforms and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Triton's programming model can target more than one backend, but source portability is only the first layer. Supported operations, code quality, tuning parameters, profiler tools, and numerical behavior can still differ by architecture.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["shared source, backend targets, architecture tuning, and unmeasured platforms"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: documented installed capability. Candidate: reviewed Triton kernel or explicit model described below.

A kernel compiled on NVIDIA is not ROCm performance evidence, even if the source contains no CUDA-specific call.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 24
LESSON_TITLE = 'Backend Portability and the ROCm Boundary'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260837
}


## 5. Freeze the experiment

**Experiment:** Record the active Triton backend and architecture while keeping the same-source ROCm claim explicitly unexecuted.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": "cuda",
  "secondary": 120,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "triton_version": "3.7.1",
    "same_source_candidate": true,
    "rocm_executed": false,
    "source_lines": 5
  }
}
This run compiled the same Triton source for backend=cuda, arch=120. ROCm portability remains unmeasured on this NVIDIA-only execution.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Active backend | cuda |
| Target architecture | 120 |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

This run compiled the same Triton source for backend=cuda, arch=120. ROCm portability remains unmeasured on this NVIDIA-only execution.

The installed toolchain or API surface was inspected. An available symbol or source file is not reported as native performance on an unexecuted backend.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Call code portable only after correctness passes on every target; call performance portable only after per-target tuning and measurement.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 24,
  "title": "Backend Portability and the ROCm Boundary",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260837
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "primary": "cuda",
    "secondary": 120,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "triton_version": "3.7.1",
      "same_source_candidate": true,
      "rocm_executed": false,
      "source_lines": 5
    }
  },
  "analysis_en": "This run compiled the same Triton source for backend=cuda, arch=120. ROCm portability remains unmeasured on this NVIDIA-only execution.",
  "analysis_zh": "本次将同一份 Triton 源码编译到 backend=cuda、arch=120；由于只运行了 NVIDIA 环境，ROCm 可移植性仍未实测。",
  "conclusion": "Call code portable only after correctness passes on every targe

## 10. Make the bounded decision

> Call code portable only after correctness passes on every target; call performance portable only after per-target tuning and measurement.

**Failure analysis:** A kernel compiled on NVIDIA is not ROCm performance evidence, even if the source contains no CUDA-specific call.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
